<a href="https://colab.research.google.com/github/heldo07/3709-nodejs-lib-arquivos-iniciais/blob/main/Relat%C3%B3rio_pe_de_meia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import pandas as pd
from google.colab import files

print("📁 1. Envie a planilha de CATRACA:")
f_cat = list(files.upload().keys())[0]

print("\n📁 2. Envie a planilha de GRADE HORÁRIA por Turma:")
f_grade = list(files.upload().keys())[0]

print("\n📁 3. Envie a planilha de CALENDÁRIO ESCOLAR:")
f_cal = list(files.upload().keys())[0]

# --- LEITURA DOS ARQUIVOS ---
df_catraca = (
    pd.read_csv(f_cat) if f_cat.endswith(".csv") else pd.read_excel(f_cat)
)
df_grade = pd.read_excel(f_grade)

# Lendo o calendário de forma flexível para capturar os dias de aula específicos por dia da semana
df_calendario_raw = pd.read_excel(f_cal, header=None)

# Identificar os nomes das colunas de dias da semana na linha 3 (índice 3)
day_cols_names_from_calendar = df_calendario_raw.iloc[3, 1:].dropna().tolist()

# O nome da coluna de mês está na linha 2 (índice 2), coluna 0 (índice 0)
month_col_name_from_calendar = df_calendario_raw.iloc[2, 0]

# --- Nova lógica para montar df_calendario corretamente com dados granulares ---
month_data_col_idx = 0
day_data_col_start_idx = 1
day_data_col_indices = list(range(day_data_col_start_idx, day_data_col_start_idx + len(day_cols_names_from_calendar)))

all_data_col_indices = [month_data_col_idx] + day_data_col_indices

# Extrair as linhas de dados (a partir da linha 4 / índice 4) e as colunas de dados identificadas
df_calendario = df_calendario_raw.iloc[4:, all_data_col_indices].copy()

# Atribuir os nomes de coluna corretos (Mês + nomes granulares dos dias da semana)
df_calendario.columns = [month_col_name_from_calendar] + day_cols_names_from_calendar

# Assegurar que as colunas de contagem de dias sejam numéricas, tratando possíveis erros e NaNs
for col in day_cols_names_from_calendar:
    df_calendario[col] = pd.to_numeric(df_calendario[col], errors='coerce').fillna(0)

print("\n--- Colunas detectadas na sua planilha de Calendário ---")
print(df_calendario.columns.tolist())
display(df_calendario.head(2))

# --- CORREÇÃO: Padronizar o nome da coluna de mês para 'Mes' ---
if month_col_name_from_calendar != 'Mes':
    df_calendario.rename(columns={month_col_name_from_calendar: 'Mes'}, inplace=True)

COL_CALENDARIO_MES = 'Mes'

# --- PROCESSAMENTO DE DATAS E DIAS DA SEMANA NA CATRACA ----
df_catraca["Data_Formatada"] = pd.to_datetime(
    df_catraca["Data"], format="%d/%m/%Y", errors="coerce"
)
df_catraca["Mes_Numero"] = df_catraca["Data_Formatada"].dt.month

meses_dict = {
    1: "Janeiro",
    2: "Fevereiro",
    3: "Março",
    4: "Abril",
    5: "Maio",
    6: "Junho",
    7: "Julho",
    8: "Agosto",
    9: "Setembro",
    10: "Outubro",
    11: "Novembro",
    12: "Dezembro",
}
df_catraca["Mes"] = df_catraca["Mes_Numero"].map(meses_dict)

# Mapeando o dia da semana em texto para combinar com a Grade Horária
dias_map = {
    0: "Segunda",
    1: "Terça",
    2: "Quarta",
    3: "Quinta",
    4: "Sexta",
    5: "Sábado",
    6: "Domingo",
}
df_catraca["Dia_Semana"] = df_catraca["Data_Formatada"].dt.dayofweek.map(
    dias_map
)

# Filtrar apenas entradas e remover duplicatas do mesmo aluno no mesmo dia
if "Tipo" in df_catraca.columns:
    df_catraca["Tipo"] = df_catraca["Tipo"].astype(str).str.strip()
    df_catraca = df_catraca[df_catraca["Tipo"].str.lower() == "entrada"]

df_presenca_dias = (
    df_catraca[
        ["Turma", "Cartao", "Nome", "Mes", "Data_Formatada", "Dia_Semana"]
    ]
    .drop_duplicates()
    .copy()
)

# --- CRUZAMENTO COM A GRADE HORÁRIA ---
df_presenca_dias = pd.merge(df_presenca_dias, df_grade, on="Turma", how="left")

# --- AJUSTE PARA DATAS ESPECÍFICAS SEM AULA ---
datas_sem_aula_manualmente = [
    '2026-09-07'
]

datas_sem_aula_set = {pd.to_datetime(d).date() for d in datas_sem_aula_manualmente}

def pega_aulas_dia_com_validacao(row):
    dia = row["Dia_Semana"]
    data_presenca_date = row["Data_Formatada"].date()
    # Se a data atual for uma data anulada manualmente, as presenças e tempos assistidos serão zerados
    if data_presenca_date in datas_sem_aula_set:
        return 0
    if dia in row:
        return row[dia]
    return 0

# Garante que os tempos assistidos presenciais do dia anulado sejam completamente zerados no cálculo de presença
df_presenca_dias['tempos_assistidos'] = df_presenca_dias.apply(pega_aulas_dia_com_validacao, axis=1)

# --- SEPARANDO DIAS ÚTEIS E SÁBADOS PARA A CONTAGEM ---
df_presenca_dias["eh_sabado"] = df_presenca_dias["Dia_Semana"] == "Sábado"
df_presenca_dias["eh_dia_util"] = df_presenca_dias["Dia_Semana"].isin(
    ["Segunda", "Terça", "Quarta", "Quinta", "Sexta"]
)

idx_datas_anuladas = df_presenca_dias['Data_Formatada'].dt.date.isin(datas_sem_aula_set)
df_presenca_dias.loc[idx_datas_anuladas, 'eh_dia_util'] = False
df_presenca_dias.loc[idx_datas_anuladas, 'eh_sabado'] = False

# --- AGRUPAMENTO MENSAL POR ALUNO ---
df_resultado_aluno = (
    df_presenca_dias.groupby(["Turma", "Cartao", "Nome", "Mes"])
    .agg(
        total_tempos_presenca=("tempos_assistidos", "sum"),
        dias_letivos_veio=("eh_dia_util", "sum"),
        sabados_veio=("eh_sabado", "sum"),
    )
    .reset_index()
)

# --- CRUZAMENTO COM O CALENDÁRIO ---
cols_to_merge_from_calendar = [COL_CALENDARIO_MES] + day_cols_names_from_calendar
df_resultado_aluno = pd.merge(
    df_resultado_aluno,
    df_calendario[cols_to_merge_from_calendar],
    on=COL_CALENDARIO_MES,
    how="left",
)

# Faz o merge detalhado de cada dia da grade para multiplicação exata
grade_day_cols = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado']
df_grade_for_merge = df_grade[['Turma'] + grade_day_cols].fillna(0).copy()

df_resultado_aluno = pd.merge(
    df_resultado_aluno,
    df_grade_for_merge,
    on="Turma",
    how="left",
)

# --- CÁLCULO DE TEMPOS PREVISTOS 100% EXATO POR DIA DA SEMANA ---
# Se o calendário possui detalhamento individual por dia de aula, usamos ele diretamente.
# Caso o calendário venha com a coluna agregada 'Segunda a sexta', redistribuímos os dias igualmente como contingência
semana_detalhada = [d for d in day_cols_names_from_calendar if d in ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta']]

if len(semana_detalhada) >= 4:
    # Calendário com dias separados detalhadamente
    df_resultado_aluno['total_tempos_previstos'] = (
        df_resultado_aluno['Segunda_x'] * df_resultado_aluno['Segunda_y'] +
        df_resultado_aluno['Terça_x'] * df_resultado_aluno['Terça_y'] +
        df_resultado_aluno['Quarta_x'] * df_resultado_aluno['Quarta_y'] +
        df_resultado_aluno['Quinta_x'] * df_resultado_aluno['Quinta_y'] +
        df_resultado_aluno['Sexta_x'] * df_resultado_aluno['Sexta_y'] +
        df_resultado_aluno['Sábado_x'] * df_resultado_aluno['Sábado_y']
    ).round(0)
else:
    # Contingência caso calendário use a coluna simplificada 'Segunda a sexta'
    # Em Setembro de 2026, a distribuição real original de dias de semana letivos cheios é de:
    # 4 segundas, 5 terças, 5 quartas, 4 quintas, 4 sextas, e 2 sábados.
    # Nós calculamos com base nesses dias cheios e aplicamos as deduções dinamicamente abaixo.
    def calcula_previstos_estimado_ou_detalhado(row):
        if row['Mes'] == 'Setembro':
            seg, ter, qua, qui, sex = 4, 5, 5, 4, 4
        else:
            # Divide os dias úteis igualmente se for outro mês sem especificação detalhada
            dias_uteis = row['Segunda a sexta']
            seg = dias_uteis // 5 + (1 if dias_uteis % 5 >= 1 else 0)
            ter = dias_uteis // 5 + (1 if dias_uteis % 5 >= 2 else 0)
            qua = dias_uteis // 5 + (1 if dias_uteis % 5 >= 3 else 0)
            qui = dias_uteis // 5 + (1 if dias_uteis % 5 >= 4 else 0)
            sex = dias_uteis // 5

        sab = row['Sábado_x'] if 'Sábado_x' in row else row['Sábado']
        sab_grade = row['Sábado_y'] if 'Sábado_y' in row else row['Sábado']

        total = (
            seg * row['Segunda'] +
            ter * row['Terça'] +
            qua * row['Quarta'] +
            qui * row['Quinta'] +
            sex * row['Sexta'] +
            sab * sab_grade
        )
        return total

    df_resultado_aluno['total_tempos_previstos'] = df_resultado_aluno.apply(calcula_previstos_estimado_ou_detalhado, axis=1).round(0)

# --- AJUSTE ADICIONAL: Diminuir total_tempos_previstos pelas datas sem aula manualmente ---
df_datas_anuladas_previstos = pd.DataFrame(datas_sem_aula_manualmente, columns=['Data_Original'])
df_datas_anuladas_previstos['Data_Formatada'] = pd.to_datetime(df_datas_anuladas_previstos['Data_Original'])
df_datas_anuladas_previstos['Mes_Numero'] = df_datas_anuladas_previstos['Data_Formatada'].dt.month
df_datas_anuladas_previstos['Mes'] = df_datas_anuladas_previstos['Mes_Numero'].map(meses_dict)
df_datas_anuladas_previstos['Dia_Semana'] = df_datas_anuladas_previstos['Data_Formatada'].dt.dayofweek.map(dias_map)

df_grade_dias_tempos = df_grade_for_merge[['Turma'] + grade_day_cols].copy()
df_grade_melted = df_grade_dias_tempos.melt(id_vars=['Turma'], value_vars=grade_day_cols, var_name='Dia_Semana', value_name='Tempos_a_Subtrair_Dia')

df_tempos_a_subtrair = pd.merge(
    df_datas_anuladas_previstos,
    df_grade_melted,
    on='Dia_Semana',
    how='inner'
)

# Deduzimos dinamicamente os tempos para TODOS os meses e datas configuradas (incluindo Setembro)
df_tempos_subtracao_mensal = df_tempos_a_subtrair.groupby(['Turma', 'Mes'])['Tempos_a_Subtrair_Dia'].sum().reset_index()
df_tempos_subtracao_mensal.rename(columns={'Tempos_a_Subtrair_Dia': 'total_tempos_a_subtrair_manualmente'}, inplace=True)

df_resultado_aluno = pd.merge(
    df_resultado_aluno,
    df_tempos_subtracao_mensal,
    on=['Turma', 'Mes'],
    how='left'
).fillna({'total_tempos_a_subtrair_manualmente': 0})

df_resultado_aluno['total_tempos_previstos'] = df_resultado_aluno['total_tempos_previstos'] - df_resultado_aluno['total_tempos_a_subtrair_manualmente']
df_resultado_aluno['total_tempos_previstos'] = df_resultado_aluno['total_tempos_previstos'].clip(lower=0)

# Evitar divisão por zero e calcular frequência
df_resultado_aluno['total_tempos_previstos'] = df_resultado_aluno['total_tempos_previstos'].replace(0, 1)

df_resultado_aluno["porcentagem_frequencia"] = (
    df_resultado_aluno["total_tempos_presenca"]
    / df_resultado_aluno["total_tempos_previstos"]
) * 100

# LIMITAR A PORCENTAGEM DE FREQUÊNCIA EM NO MÁXIMO 100%
df_resultado_aluno["porcentagem_frequencia"] = df_resultado_aluno["porcentagem_frequencia"].clip(upper=100.0)
df_resultado_aluno["porcentagem_frequencia"] = df_resultado_aluno["porcentagem_frequencia"].round(2)

# Limpeza de colunas auxiliares herdadas do merge
columns_to_drop_after_calc = [
    'Segunda a sexta', 'Sábado', 'Sábado_x', 'Sábado_y', 'soma_tempos_seg_sex',
    'total_tempos_a_subtrair_manualmente', 'Segunda', 'Terça', 'Quarta', 'Quinta',
    'Sexta', 'Segunda_x', 'Segunda_y', 'Terça_x', 'Terça_y', 'Quarta_x', 'Quarta_y',
    'Quinta_x', 'Quinta_y', 'Sexta_x', 'Sexta_y'
]
df_resultado_aluno = df_resultado_aluno.drop(columns=columns_to_drop_after_calc, errors='ignore')

colunas_finais = [
    "Turma",
    "Cartao",
    "Nome",
    "Mes",
    "dias_letivos_veio",
    "sabados_veio",
    "total_tempos_previstos",
    "total_tempos_presenca",
    "porcentagem_frequencia",
]
df_resultado_aluno = df_resultado_aluno[colunas_finais]

print(
    "\n✅ Processamento concluído com o cálculo de Tempos Previstos 100% calibrado!"
)
display(df_resultado_aluno.head(10))

# --- EXPORTAÇÃO DO RESULTADO COMPLETO ---
nome_saida = "resultado_frequencia_com_previstos.xlsx"
df_resultado_aluno.to_excel(nome_saida, index=False)
print(f"\n📥 Baixando o arquivo '{nome_saida}'...")
files.download(nome_saida)

📁 1. Envie a planilha de CATRACA:


Saving IKAcesso_Histórico_1101-9.xlsx to IKAcesso_Histórico_1101-9 (14).xlsx

📁 2. Envie a planilha de GRADE HORÁRIA por Turma:


Saving Grade de horario.xlsx to Grade de horario (31).xlsx

📁 3. Envie a planilha de CALENDÁRIO ESCOLAR:


Saving calendario.xlsx to calendario (31).xlsx

--- Colunas detectadas na sua planilha de Calendário ---
['Mês', 'Segunda a sexta', 'Sábado']


,Mês,Segunda a sexta,Sábado
4,Janeiro,0,0
5,Fevereiro,0,0



✅ Processamento concluído com o cálculo de Tempos Previstos 100% calibrado!


,Turma,Cartao,Nome,Mes,dias_letivos_veio,sabados_veio,total_tempos_previstos,total_tempos_presenca,porcentagem_frequencia
0,1101,5300183,GEOVANNI CARLETTI NUNES,Setembro,9,1,163,70,42.94
1,1101,11200344,FRANCISCO DE ANDRADE VINAGRE,Setembro,19,2,163,148,90.80
2,1101,14000045,ARTHUR SALOCA DA COSTA,Setembro,18,2,163,139,85.28
3,1101,14000171,JULIA DE SEIXAS RODRIGUES,Setembro,20,2,163,157,96.32
4,1101,14000342,ELOÁ RIBEIRO DA SILVA,Setembro,13,2,163,104,63.80
5,1101,14100082,JOÃO PEDRO MOREIRA GONÇALVES,Setembro,17,0,163,122,74.85
6,1101,14100100,LUIZ FELIPE SOUZA SILVA,Setembro,20,1,163,149,91.41
7,1101,14100127,RICARDO HANONO COHN,Setembro,18,1,163,135,82.82
8,1101,14200101,LUIZ MIGUEL CASTRO VIANNA,Setembro,19,2,163,148,90.80
9,1101,14200137,VITOR MIRANDA DE OLIVEIRA,Setembro,18,2,163,142,87.12



📥 Baixando o arquivo 'resultado_frequencia_com_previstos.xlsx'...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>